# 01 - Cleaning data

Consolidates the cleaning/transformation steps decided during Data Collection &
EDA, so they're reproducible in one place rather than scattered across the
per-file EDA notebooks. Reads from `data/raw/`, writes cleaned outputs to
`data/processed/`. Decisions behind each step are logged in
`docs/kg_modelling_decisions.md`.

Three transformations, one per POI source that needed one:
1. **Museen** - remove one exact duplicate row
2. **Parkanlagen** - parse `FLAECHE` (area) from formatted string to numeric
3. **Wien Tourismus** - extract the "Sehenswürdigkeit" + "Schloss & Palais"
   subcategories as a 7th POI source, derive district from postal code, remove
   duplicate landmark listings

Büchereien, Badestellen, Schwimmbäder, and Spielplätze needed no cleaning and are
used directly from `data/raw/`.

In [1]:
import pandas as pd
import re

RAW = "../data/raw"
PROCESSED = "../data/processed" 

## 1. Museen - remove duplicate row

`MUSEUMOGD.csv` EDA found one exact duplicate: "MAK - Museum für angewandte
Kunst" appears twice with identical district and address. Dedup on
`NAME`+`BEZIRK`+`ADRESSE` (safer than a full-row dedup, since a coincidental
full-row match isn't required for this to be the same real-world museum listed
twice).

In [2]:
museum = pd.read_csv(f"{RAW}/MUSEUMOGD.csv")
before = len(museum)

museum_clean = museum.drop_duplicates(subset=["NAME", "BEZIRK", "ADRESSE"], keep="first")
after = len(museum_clean)

print(f"MUSEUMOGD: {before} -> {after} rows")
museum_clean.to_csv(f"{PROCESSED}/MUSEUMOGD_clean.csv", index=False)

MUSEUMOGD: 137 -> 136 rows


## 2. Parkanlagen - parse `FLAECHE` to numeric

`FLAECHE` is a formatted string like `"14.714 m²"` (German thousands-separator:
`.` groups thousands, no decimal comma used here). Strip the unit, remove the
thousands separator, and cast to float as `FLAECHE_M2`.

In [3]:
parks = pd.read_csv(f"{RAW}/PARKINFOOGD.csv")

def parse_flaeche(s):
    if pd.isnull(s):
        return None
    m = re.match(r"([\d.,]+)\s*m", str(s))
    if not m:
        return None
    num = m.group(1).replace(".", "").replace(",", ".")
    try:
        return float(num)
    except ValueError:
        return None

parks["FLAECHE_M2"] = parks["FLAECHE"].map(parse_flaeche)

print("rows:", len(parks), "| unparsed FLAECHE:", parks["FLAECHE_M2"].isnull().sum())
print(parks[["ANL_NAME", "FLAECHE", "FLAECHE_M2"]].sample(5, random_state=1))
print("\nFLAECHE_M2 describe:")
print(parks["FLAECHE_M2"].describe())

parks.to_csv(f"{PROCESSED}/PARKINFOOGD_clean.csv", index=False)

rows: 1051 | unparsed FLAECHE: 0
                       ANL_NAME     FLAECHE  FLAECHE_M2
49             PA Kirschenallee   14.714 m²     14714.0
741             GA Apostelgasse      501 m²       501.0
358             Hans-Moser-Park    1.142 m²      1142.0
993  Prater - Sonnenscheinwiese  141.396 m²    141396.0
639         Fritzi-Massary-Park    4.747 m²      4747.0

FLAECHE_M2 describe:
count      1051.000000
mean      12554.622265
std       43142.281635
min           0.000000
25%        1216.500000
50%        3447.000000
75%        7994.500000
max      634043.000000
Name: FLAECHE_M2, dtype: float64


## 3. Wien Tourismus - extract sights subcategories

Pull just `Sehenswürdigkeit` and `Schloss & Palais` out of
`WIENTOURISMUSOGD.csv` (250 of the file's 2,860 rows) as a 7th POI source,
rather than ingesting the whole tourism file (which is dominated by
restaurants/hotels/bars, out of scope).

Three sub-steps:
- derive `BEZIRK` from `POSTALCODE` (the file has no district column) using
  Vienna's postal code convention: `BEZIRK = (POSTALCODE - 1000) / 10`, valid for
  1010–1230 in steps of 10
- parse `SHAPE` into `lon`/`lat`
- remove duplicate landmark listings (same name+street, ~10-15m coordinate
  drift, different `UID_` - genuine duplicate listings, unlike Spielplätze's
  repeated names which are distinct sub-features and were deliberately *not*
  deduplicated, see `docs/kg_modelling_decisions.md`)

In [4]:
tourism = pd.read_csv(f"{RAW}/WIENTOURISMUSOGD.csv")
sights = tourism[tourism["SUBCATEGORY_NAME"].isin(["Sehenswürdigkeit", "Schloss & Palais"])].copy()

print("extracted:", sights.shape)
print(sights["SUBCATEGORY_NAME"].value_counts())

extracted: (250, 15)
SUBCATEGORY_NAME
Sehenswürdigkeit    207
Schloss & Palais     43
Name: count, dtype: int64


In [5]:
def postcode_to_bezirk(pc):
    if pd.isnull(pc):
        return None
    pc = int(pc)
    if 1010 <= pc <= 1230 and pc % 10 == 0:
        return (pc - 1000) // 10
    return None

sights["BEZIRK"] = sights["POSTALCODE"].map(postcode_to_bezirk)
print("unmapped postcodes:", sights[sights["BEZIRK"].isnull()]["POSTALCODE"].tolist())
print("\nBEZIRK distribution:")
print(sights["BEZIRK"].value_counts(dropna=False).sort_index())

unmapped postcodes: []

BEZIRK distribution:
BEZIRK
1     89
2     21
3     17
4     10
5      8
6      9
7      5
8      1
9      9
10     8
11     6
12     3
13    12
14     3
15     6
16     4
17     1
18     3
19    15
20     1
21     4
22    13
23     2
Name: count, dtype: int64


In [6]:
def parse_point(s):
    m = re.search(r"(-?\d+\.\d+)\s+(-?\d+\.\d+)", str(s))
    return (float(m.group(1)), float(m.group(2))) if m else (None, None)

sights["lon"], sights["lat"] = zip(*sights["SHAPE"].map(parse_point))
print("unparseable SHAPE:", sights["lon"].isnull().sum())

unparseable SHAPE: 0


In [7]:
dup_names = sights[sights.duplicated("NAME", keep=False)].sort_values("NAME")
print("rows sharing a NAME before dedup:")
print(dup_names[["NAME", "STREET", "UID_", "lon", "lat"]])

before = len(sights)
sights_clean = sights.drop_duplicates(subset=["NAME", "STREET"], keep="first")
after = len(sights_clean)
print(f"\n{before} -> {after} rows after removing genuine duplicate landmarks")

sights_clean.to_csv(f"{PROCESSED}/WIENTOURISMUS_sights_clean.csv", index=False)

rows sharing a NAME before dedup:
                 NAME                                  STREET  \
1592  Haus des Meeres  Fritz-Grünbaum-Platz 1 (Esterházypark)   
2429  Haus des Meeres  Fritz-Grünbaum-Platz 1 (Esterházypark)   
1353       Judenplatz                              Judenplatz   
2430       Judenplatz                              Judenplatz   
1413     Rathausplatz                            Rathausplatz   
2437     Rathausplatz                            Rathausplatz   

                                  UID_        lon        lat  
1592  92a0da76d0b67da200eb9f4ab5309748  16.352916  48.197607  
2429  59406f83fbeffd8b4b0070cdf70c465c  16.353034  48.197557  
1353  4202a7e6b9b4fbd85b477f19d0b93600  16.369564  48.211523  
2430  405ff3efbdaf2241dde16062fc1764ad  16.369629  48.211615  
1413  62d97e81314261f35e8489ea08cd4fd8  16.358863  48.210515  
2437  0c4cbf50430de20547a6dd8bee05128f  16.359426  48.210314  

250 -> 247 rows after removing genuine duplicate landmarks


## Summary

Files written to `data/processed/`:

| File | Rows | Transformation |
|---|---|---|
| `MUSEUMOGD_clean.csv` | 136 | 1 exact duplicate removed |
| `PARKINFOOGD_clean.csv` | 1,051 | `FLAECHE_M2` numeric column added |
| `WIENTOURISMUS_sights_clean.csv` | 247 | Filtered to 2 subcategories, `BEZIRK` derived from postal code, 3 duplicate landmarks removed |

Büchereien, Badestellen, Schwimmbäder, Spielplätze needed no cleaning - used
directly from `data/raw/` in later steps. Full rationale for each decision is in
`docs/kg_modelling_decisions.md`.